# Multiple Linear Regression (Baseline)

Notebook ini menggunakan **data clean** tanpa feature engineering. Tujuannya hanya melihat performa baseline awal.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def find_clean_data():
    roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    candidates = [r / 'data/processed data/ali_f_event_clean.csv' for r in roots]
    for p in candidates:
        if p.exists():
            return p, candidates
    return None, candidates

data_path, checked = find_clean_data()
if data_path is None:
    print('CWD:', Path.cwd())
    print('Checked paths:')
    for p in checked:
        print(' -', p)
    raise FileNotFoundError('Clean data not found in expected locations.')

df = pd.read_csv(data_path, parse_dates=['Date'])
df = df.set_index('Date').sort_index()

print(f'Data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Periode: {df.index.min().date()} s/d {df.index.max().date()}')


Data loaded: 2386 rows × 19 columns
Periode: 2014-05-06 s/d 2026-03-04


In [2]:
# Pilih fitur dasar (tanpa feature engineering)
feature_candidates = ['Open', 'High', 'Low', 'Volume']
feature_cols = [c for c in feature_candidates if c in df.columns]
if not feature_cols:
    feature_cols = df.select_dtypes(include='number').columns.difference(['Close', 'Return']).tolist()

target_col = 'Close'

# Drop baris yang ada NA di fitur/target
df_model = df[feature_cols + [target_col]].dropna().copy()

# Split time-based: 70% train, 15% val, 15% test
n = len(df_model)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df_model.iloc[:train_end]
val = df_model.iloc[train_end:val_end]
test = df_model.iloc[val_end:]

X_train, y_train = train[feature_cols], train[target_col]
X_val, y_val = val[feature_cols], val[target_col]
X_test, y_test = test[feature_cols], test[target_col]

print('Feature columns:', feature_cols)
print('Train/Val/Test sizes:', len(train), len(val), len(test))


Feature columns: ['Open', 'High', 'Low', 'Volume']
Train/Val/Test sizes: 1670 358 358


In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

def eval_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

metrics = pd.DataFrame([
    eval_metrics(y_train, y_pred_train),
    eval_metrics(y_val, y_pred_val),
    eval_metrics(y_test, y_pred_test)
], index=['Train', 'Validation', 'Test'])
metrics


,MSE,RMSE,MAE,R2
Train,3.810647,1.952088,0.555156,0.999978
Validation,9.537029,3.088208,1.004906,0.999470
Test,20.174239,4.491574,1.541097,0.999538
